# 실습 2: 협업 필터링 Explained (한 줄씩 분해하기)

이 실습은 `lab_01`에서 실행했던 파이프라인을 **한 줄씩 분해**하여 각 코드가 어떤 원리로 동작하는지 이해하는 것을 목표로 합니다.

**개념 복기 및 이론 점검**
- 코사인 유사도는 두 벡터의 방향이 얼마나 비슷한지를 재는 수치입니다. 평점 벡터에서 '방향이 비슷하다'는 것은 무엇을 의미하는가?
- `torch.mm(normalized, normalized.T)`가 왜 모든 사용자 쌍의 코사인 유사도를 한 번에 계산하는가?
- User-based CF와 Item-based CF 중 대규모 서비스에서 더 많이 쓰이는 방식은 무엇이고 왜인가?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Team-AnI/A-AND-I-4TH-AI-CODE-LAB/blob/main/4주차/lab_02_explain.ipynb)

> 위 배지를 누르면 이 노트북이 **여러분 Google 계정의 Colab**에서 열립니다. 수정본을 남기려면 `파일 → Drive에 사본 저장`.

## 1. 환경 준비: 라이브러리 임포트

In [ ]:
import pandas as pd
import torch

torch.manual_seed(42)

### 🔬 코드 해설
- `pandas`: 테이블 형태의 데이터를 다루는 라이브러리입니다. CSV 파일 로딩과 딕셔너리 변환에 사용합니다.
- `torch`: 텐서 연산과 행렬 곱 기반 유사도 계산에 사용합니다.
- `torch.manual_seed(42)`: 난수 시드를 고정해 실험의 재현성을 보장합니다.

## 2. 데이터 다운로드

In [ ]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-100k.zip -O ml-100k.zip
!unzip -q -o ml-100k.zip
print("다운로드 완료")

### 🔬 코드 해설
- **MovieLens ml-100k**: GroupLens 연구소에서 수집한 공개 영화 평점 데이터셋입니다. 943명의 사용자가 1682편의 영화에 매긴 100,000개의 평점을 담고 있습니다.
- **`u.data`**: 탭(\t)으로 구분된 `user_id | item_id | rating | timestamp` 형식의 파일입니다.
- **`u.item`**: 영화 ID와 영화 제목 등 메타데이터를 담은 파일입니다. 추천 결과를 사람이 읽을 수 있는 형태로 출력할 때 사용합니다.

## 3. User-Item 평점 행렬 구성

In [ ]:
df = pd.read_csv(
    'ml-100k/u.data', sep='\t', header=None,
    names=['user_id', 'item_id', 'rating', 'timestamp']
)
movies = pd.read_csv(
    'ml-100k/u.item', sep='|', header=None, encoding='latin-1',
    usecols=[0, 1], names=['item_id', 'title']
)
movie_names = dict(zip(movies['item_id'], movies['title']))

n_users = df['user_id'].max()
n_items = df['item_id'].max()

ratings = torch.zeros(n_users, n_items)
for row in df.itertuples():
    ratings[row.user_id - 1, row.item_id - 1] = row.rating

print(f"행렬 크기: {ratings.shape}  (사용자 × 영화)")

### 🔬 코드 해설
- **`pd.read_csv('ml-100k/u.data', sep='\t', header=None, ...)`**: 평점 파일을 불러옵니다. `sep='\t'`는 탭 구분자, `header=None`은 첫 줄에 헤더가 없음을 의미합니다.
- **`dict(zip(movies['item_id'], movies['title']))`**: 영화 ID를 키, 제목을 값으로 하는 딕셔너리를 만듭니다. 추천 결과 출력 시 ID 대신 영화 제목을 표시하기 위해 사용합니다.
- **`torch.zeros(n_users, n_items)`**: 943 × 1682 크기의 0으로 채워진 행렬을 만듭니다. 0은 '아직 평가하지 않음'을 의미합니다.
- **`ratings[row.user_id - 1, row.item_id - 1] = row.rating`**: ml-100k의 ID는 1부터 시작하므로 `-1`을 빼서 0-indexed 텐서 인덱스로 맞춥니다.
- 결과 행렬의 각 **행**은 한 사용자의 평점 벡터, 각 **열**은 한 영화에 대한 모든 사용자의 평점 벡터입니다. 대부분의 칸이 0인 이 구조를 **희소 행렬(Sparse Matrix)** 이라고 합니다.

## 4. 코사인 유사도 계산

In [ ]:
norms = torch.norm(ratings, dim=1, keepdim=True).clamp(min=1e-8)
normalized = ratings / norms
user_sim = torch.mm(normalized, normalized.T)  # (943, 943)

print(f"사용자 유사도 행렬 크기: {user_sim.shape}")
print(f"사용자 0과 사용자 1의 유사도: {user_sim[0, 1].item():.4f}")

### 🔬 코드 해설
코사인 유사도 공식은 `cos(a, b) = (a · b) / (‖a‖ × ‖b‖)` 입니다. 이를 두 단계로 계산합니다.

- **`torch.norm(ratings, dim=1, keepdim=True)`**: 각 사용자 평점 벡터의 크기(L2 norm)를 계산합니다. `dim=1`은 행 방향, 즉 각 사용자별로 계산하라는 의미입니다. `keepdim=True`는 결과 형태를 `(943, 1)`로 유지해 이후 나눗셈에서 브로드캐스팅이 올바르게 동작하도록 합니다.
- **`.clamp(min=1e-8)`**: 한 번도 평점을 매기지 않은 사용자의 벡터 크기가 0이 되어 나눗셈 오류가 나는 것을 방지합니다.
- **`ratings / norms`**: 각 벡터를 자신의 크기로 나누면 방향만 남고 크기가 1인 **단위 벡터**가 됩니다. 이를 정규화(Normalization)라고 합니다.
- **`torch.mm(normalized, normalized.T)`**: 단위 벡터끼리의 내적(dot product)은 곧 코사인 유사도입니다. 행렬 곱 `A @ A^T`는 `A`의 모든 행 쌍의 내적을 한 번에 계산하므로, `result[i][j]`가 사용자 i와 j의 코사인 유사도가 됩니다. 반복문 없이 한 번의 연산으로 처리합니다.

## 5. User-based CF

In [ ]:
def recommend_user_based(user_idx, ratings, user_sim, top_n_neighbors=20, top_n_items=10):
    sim_scores = user_sim[user_idx].clone()
    sim_scores[user_idx] = -1

    top_neighbors = torch.topk(sim_scores, top_n_neighbors).indices

    unrated_mask = (ratings[user_idx] == 0)
    neighbor_ratings = ratings[top_neighbors]           # (top_n_neighbors, n_items)
    weights = sim_scores[top_neighbors].unsqueeze(1)    # (top_n_neighbors, 1)

    scores = (neighbor_ratings * weights).sum(dim=0)    # (n_items,)
    scores[~unrated_mask] = -float('inf')

    return torch.topk(scores, top_n_items).indices.tolist()

### 🔬 코드 해설
- **`user_sim[user_idx].clone()`**: 원본 `user_sim` 행렬을 수정하지 않기 위해 복사본을 만듭니다. PyTorch에서 텐서 슬라이싱은 뷰(view)를 반환하므로 복사 없이 수정하면 원본이 바뀝니다.
- **`sim_scores[user_idx] = -1`**: 자기 자신과의 유사도는 항상 1(최댓값)이므로, -1로 설정해 이웃 선정에서 제외합니다.
- **`torch.topk(sim_scores, top_n_neighbors).indices`**: 유사도 값이 가장 높은 상위 k명의 인덱스를 반환합니다. 이 k명이 이웃(neighbor)입니다. k는 하이퍼파라미터로, 너무 적으면 추천이 편향되고 너무 많으면 유사도가 낮은 사용자까지 포함됩니다.
- **`neighbor_ratings * weights`**: 유사도가 높은 이웃의 평점에 더 높은 가중치를 곱합니다. `weights`의 형태가 `(k, 1)`이므로 `neighbor_ratings (k, n_items)`와 브로드캐스팅으로 곱해집니다.
- **`.sum(dim=0)`**: 모든 이웃의 가중 평점을 영화별로 합산합니다. `dim=0`은 행 방향으로 합산한다는 의미입니다.
- **`scores[~unrated_mask] = -float('inf')`**: 이미 본 영화의 점수를 음의 무한대로 설정해 `topk` 결과에 절대 등장하지 않도록 합니다.

## 6. 추천 생성

In [ ]:
user_idx = 0
ub_rec_items = recommend_user_based(user_idx, ratings, user_sim)

print(f"사용자 {user_idx + 1}번의 User-based CF 추천 영화 Top 10:")
for rank, item_idx in enumerate(ub_rec_items, 1):
    print(f"  {rank:2d}. {movie_names.get(item_idx + 1, 'Unknown')}")

### 🔬 코드 해설
- **`user_idx = 0`**: 0-indexed로 0번은 데이터셋의 1번 사용자입니다.
- **`movie_names.get(item_idx + 1, 'Unknown')`**: 텐서 인덱스는 0-based이므로 `+1`을 더해 MovieLens의 1-based ID로 변환한 뒤 영화 제목을 조회합니다. 딕셔너리에 없는 ID가 들어올 경우 'Unknown'을 기본값으로 반환합니다.

> 💡 **직접 해보기**: `user_idx`를 다른 값으로 바꾸거나, `top_n_neighbors`를 5 또는 50으로 바꾸면 추천 결과가 어떻게 달라지는지 확인해보세요.

## 7. Item-based CF와의 비교

In [ ]:
item_norms = torch.norm(ratings.T, dim=1, keepdim=True).clamp(min=1e-8)
item_normalized = ratings.T / item_norms
item_sim = torch.mm(item_normalized, item_normalized.T)  # (1682, 1682)

def recommend_item_based(user_idx, ratings, item_sim, top_n=10):
    user_ratings = ratings[user_idx]
    unrated_mask = (user_ratings == 0)
    scores = torch.mv(item_sim, user_ratings)
    scores[~unrated_mask] = -float('inf')
    return torch.topk(scores, top_n).indices.tolist()

ib_rec_items = recommend_item_based(user_idx, ratings, item_sim)

print(f"사용자 {user_idx + 1}번의 Item-based CF 추천 영화 Top 10:")
for rank, item_idx in enumerate(ib_rec_items, 1):
    print(f"  {rank:2d}. {movie_names.get(item_idx + 1, 'Unknown')}")

ub_set = set(ub_rec_items)
ib_set = set(ib_rec_items)
print(f"\n겹치는 영화 ({len(ub_set & ib_set)}편):")
for item_idx in ub_set & ib_set:
    print(f"  - {movie_names.get(item_idx + 1, 'Unknown')}")

### 🔬 코드 해설
- **`ratings.T`**: User-based CF에서는 `ratings`의 각 **행**을 사용자 벡터로 사용했습니다. Item-based CF에서는 `ratings.T`의 각 **행**, 즉 원본 행렬의 각 **열**을 영화 벡터로 사용합니다. 코드 구조는 완전히 동일하고, 입력만 전치(transpose)한 것입니다.
- **`torch.mv(item_sim, user_ratings)`**: 행렬-벡터 곱입니다. `item_sim`의 각 행 i는 영화 i와 모든 영화의 유사도 벡터입니다. 이 벡터와 `user_ratings`(사용자의 평점 벡터)의 내적은 '영화 i와 사용자가 높게 평가한 영화들의 유사도 가중합'이 됩니다. 즉, `scores[i]`가 높을수록 사용자가 좋아할 만한 영화입니다.

| | User-based CF | Item-based CF |
|---|---|---|
| 유사도 계산 대상 | 사용자 간 | 아이템 간 |
| 추천 아이디어 | 나와 비슷한 사람들이 좋아한 것 | 내가 좋아했던 것과 비슷한 것 |
| 유사도 행렬 크기 | 사용자 수 × 사용자 수 | 아이템 수 × 아이템 수 |
| 사전 계산 용이성 | 사용자 변화가 잦아 어려움 | 아이템이 상대적으로 안정적 → 가능 |
| 실제 사례 | - | 아마존 "함께 구매한 상품" |

## 8. 학습 결과 정리
- `lab_01`의 전체 코드를 **환경 준비 → 행렬 구성 → 유사도 계산 → CF 추천 → 비교**의 각 단계로 나누어 원리를 해부했습니다.
- 정규화 후 행렬 곱(`torch.mm`)으로 모든 쌍의 코사인 유사도를 한 번에 계산하는 원리를 이해했습니다.
- User-based와 Item-based CF의 코드 구조가 `ratings`와 `ratings.T`의 차이 하나로 대칭됨을 확인했습니다.